# Phase 2b — Baseline vs Expanded Feature Set

Same model class (GBM Stack: XGB + LGBM + CatBoost + LR meta), two feature sets:

1. **Baseline** — `data/features/baseline_features_*.parquet` (~30 columns)
2. **Expanded** — `data/features/expanded_features_*.parquet` + OOF target encoding on `last_country` and `cohort_month` (~75 columns)

Both runs are logged to MLflow under experiment `churn-fe-comparison`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.metrics import roc_auc_score  # noqa: F401  (handy in notebook)

from src.models.churn.classification.stack import ChurnStack
from src.evaluation.churn_metrics import evaluate
from src.features.target_encoding import fit_oof

PROC = PROJECT_ROOT / 'data' / 'processed'
FEAT = PROJECT_ROOT / 'data' / 'features'
REPORTS = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

labels = {n: pd.read_parquet(PROC / f'churn_labels_{n}.parquet') for n in ['train','val','test']}
mlflow.set_experiment('churn-fe-comparison')

## Load both feature sets and join with labels

In [ ]:
def load_and_join(kind: str):
    out = {}
    for name in ['train','val','test']:
        f = pd.read_parquet(FEAT / f'{kind}_features_{name}.parquet')
        out[name] = f.merge(labels[name][['customer_id','churn']], on='customer_id', how='inner')
    return out

base = load_and_join('baseline')
expd = load_and_join('expanded')
for name in ['train','val','test']:
    print(f'{name:>5}  baseline {base[name].shape}  expanded {expd[name].shape}  churn={base[name]["churn"].mean():.3f}')

## Apply OOF target encoding to the expanded set

In [ ]:
# Encode last_country and cohort_month (low-cardinality, easy wins)
for group in ['last_country', 'cohort_month']:
    if group not in expd['train'].columns:
        continue
    oof_train, encoder = fit_oof(expd['train'], 'churn', group, n_folds=5)
    expd['train'][f'te_{group}'] = oof_train.values
    for name in ['val','test']:
        expd[name][f'te_{group}'] = encoder.transform(expd[name]).values
    print(f'Target-encoded {group}: train mean={expd["train"][f"te_{group}"].mean():.3f}, '
          f'val mean={expd["val"][f"te_{group}"].mean():.3f}, '
          f'test mean={expd["test"][f"te_{group}"].mean():.3f}')

## Helper: pick numeric feature columns

In [ ]:
def numeric_cols(df, drop=('customer_id','churn','last_country','country')):
    return [c for c in df.columns if c not in drop and pd.api.types.is_numeric_dtype(df[c])]

base_cols = numeric_cols(base['train'])
expd_cols = numeric_cols(expd['train'])
print(f'baseline: {len(base_cols)} features\nexpanded: {len(expd_cols)} features')

## Train + evaluate each run

In [ ]:
def run(name, data, cols):
    Xtr, ytr = data['train'][cols].fillna(0), data['train']['churn']
    Xva, yva = data['val'][cols].fillna(0), data['val']['churn']
    Xte, yte = data['test'][cols].fillna(0), data['test']['churn']

    with mlflow.start_run(run_name=name):
        mlflow.log_param('n_features', len(cols))
        mlflow.log_param('feature_set', name)
        stack = ChurnStack().fit(Xtr, ytr)
        p_va = stack.predict_proba(Xva)
        p_te = stack.predict_proba(Xte)
        m_va = evaluate(yva.values, p_va)
        m_te = evaluate(yte.values, p_te, threshold=m_va['threshold'])
        for k, v in m_va.items():
            if isinstance(v, float): mlflow.log_metric(f'val_{k}', v)
        for k, v in m_te.items():
            if isinstance(v, float): mlflow.log_metric(f'test_{k}', v)
    return stack, m_va, m_te, p_te

stack_b, mv_b, mt_b, p_test_b = run('baseline', base, base_cols)
stack_e, mv_e, mt_e, p_test_e = run('expanded', expd, expd_cols)

## Side-by-side comparison

In [ ]:
def row(name, m_val, m_test):
    return {
        'set': name,
        'val_auc': m_val['auc_roc'], 'val_pr_auc': m_val['pr_auc'], 'val_brier': m_val['brier_score'], 'val_f1': m_val['f1'],
        'test_auc': m_test['auc_roc'], 'test_pr_auc': m_test['pr_auc'], 'test_brier': m_test['brier_score'], 'test_f1': m_test['f1'],
    }

table = pd.DataFrame([row('baseline', mv_b, mt_b), row('expanded', mv_e, mt_e)]).set_index('set').round(4)
delta = (table.loc['expanded'] - table.loc['baseline']).round(4)
delta.name = 'delta (expanded - baseline)'
table = pd.concat([table, delta.to_frame().T])
table.to_csv(REPORTS / 'fe_comparison.csv')
table

## Feature importance on the expanded model

Pull importances from the first fold of the LightGBM base learner — cheap proxy for which families matter.

In [ ]:
lgbm_models = [m for name, ms in stack_e.base_models for m in ms if name == 'lgbm']
imp = pd.DataFrame({
    'feature': expd_cols,
    'gain': np.mean([m.booster_.feature_importance(importance_type='gain') for m in lgbm_models], axis=0),
}).sort_values('gain', ascending=False)
imp.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
imp.head(25).iloc[::-1].plot.barh(x='feature', y='gain', ax=ax, legend=False)
ax.set_title('LightGBM gain — top 25 features (expanded set)')
plt.tight_layout()
plt.savefig(REPORTS / 'fe_top25_importances.png', dpi=120)
plt.show()